In [20]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline

In [21]:
from scipy.signal import welch, hilbert
from scipy.stats import linregress

from itertools import combinations


# ============================================================
# PATHS
# ============================================================

DATA_ROOT = Path(
    r"C:\Users\Sanjeev\OneDrive\Desktop\Brain ID\EEG\ds004148"
)

PROCESSED_ROOT = DATA_ROOT / "processed_windows"

OUTPUT_FILE = (
    PROCESSED_ROOT / "combined_eeg_features.csv"
)


# ============================================================
# EEG SETTINGS
# ============================================================

CHANNELS = [
    "Fp1",
    "Fp2",
    "Fz"
]

SFREQ = 500.0

# Frequency bands
BANDS = {
    "delta": (1.0, 4.0),
    "theta": (4.0, 8.0),
    "alpha": (8.0, 13.0),
    "beta":  (13.0, 30.0),
    "gamma": (30.0, 45.0)
}

TOTAL_BAND = (1.0, 45.0)

In [22]:
def compute_psd(signal, sfreq):
    """
    Compute Welch PSD.
    """

    freqs, psd = welch(
        signal,
        fs=sfreq,
        nperseg=min(1000, len(signal)),
        noverlap=500 if len(signal) >= 1000 else 0,
        window="hann"
    )

    return freqs, psd


def bandpower(freqs, psd, low, high):
    """
    Absolute bandpower using integration.
    """

    mask = (
        (freqs >= low) &
        (freqs < high)
    )

    if not np.any(mask):
        return 0.0

    return np.trapezoid(
        psd[mask],
        freqs[mask]
    )


def safe_log(x, eps=1e-12):
    """
    Numerically stable natural logarithm.
    """

    return np.log(
        np.maximum(x, eps)
    )

In [23]:
def hjorth_parameters(signal):
    """
    Hjorth activity, mobility and complexity.
    """

    signal = np.asarray(signal)

    variance = np.var(signal)

    if variance < 1e-12:
        return 0.0, 0.0

    first_derivative = np.diff(signal)

    second_derivative = np.diff(
        first_derivative
    )

    var_d1 = np.var(first_derivative)

    var_d2 = np.var(second_derivative)

    mobility = np.sqrt(
        var_d1 / variance
    )

    if var_d1 < 1e-12:
        complexity = 0.0
    else:
        mobility_d1 = np.sqrt(
            var_d2 / var_d1
        )

        complexity = (
            mobility_d1 / mobility
        )

    return mobility, complexity

In [24]:
def sample_entropy(
    signal,
    m=2,
    r_ratio=0.2
):
    """
    Sample entropy.

    m:
        embedding dimension

    r_ratio:
        tolerance as a fraction of signal std
    """

    x = np.asarray(signal)

    n = len(x)

    if n < 100:
        return np.nan

    std = np.std(x)

    if std < 1e-12:
        return 0.0

    r = r_ratio * std

    def count_matches(m):
        templates = np.array([
            x[i:i+m]
            for i in range(n - m)
        ])

        count = 0

        for i in range(len(templates)):

            distances = np.max(
                np.abs(
                    templates[i+1:]
                    - templates[i]
                ),
                axis=1
            )

            count += np.sum(
                distances <= r
            )

        return count

    B = count_matches(m)
    A = count_matches(m + 1)

    if B == 0 or A == 0:
        return np.nan

    return -np.log(
        A / B
    )

In [25]:
def aperiodic_features(
    freqs,
    psd,
    low=2.0,
    high=40.0
):
    """
    Estimate aperiodic slope and offset
    using linear regression in log-log space.
    """

    mask = (
        (freqs >= low) &
        (freqs <= high) &
        (psd > 0)
    )

    if np.sum(mask) < 10:
        return np.nan, np.nan

    log_freq = np.log10(
        freqs[mask]
    )

    log_power = np.log10(
        psd[mask]
    )

    result = linregress(
        log_freq,
        log_power
    )

    slope = result.slope
    offset = result.intercept

    return slope, offset

In [26]:
def bandpass_fft_signal(
    signal,
    sfreq,
    low,
    high
):
    """
    Simple FFT-based bandpass filtering.

    Used only for phase extraction.
    """

    n = len(signal)

    freqs = np.fft.rfftfreq(
        n,
        d=1 / sfreq
    )

    fft_signal = np.fft.rfft(
        signal
    )

    mask = (
        (freqs >= low) &
        (freqs <= high)
    )

    filtered_fft = (
        fft_signal * mask
    )

    return np.fft.irfft(
        filtered_fft,
        n=n
    )


def compute_plv(
    signal1,
    signal2,
    sfreq,
    low,
    high
):

    x1 = bandpass_fft_signal(
        signal1,
        sfreq,
        low,
        high
    )

    x2 = bandpass_fft_signal(
        signal2,
        sfreq,
        low,
        high
    )

    phase1 = np.angle(
        hilbert(x1)
    )

    phase2 = np.angle(
        hilbert(x2)
    )

    phase_difference = (
        phase1 - phase2
    )

    plv = np.abs(
        np.mean(
            np.exp(
                1j * phase_difference
            )
        )
    )

    return plv

In [27]:
def extract_features(
    window,
    sfreq
):

    features = {}

    # ========================================================
    # PSD + BANDPOWER
    # ========================================================

    psds = {}

    for ch_idx, ch_name in enumerate(CHANNELS):

        signal = window[ch_idx]

        freqs, psd = compute_psd(
            signal,
            sfreq
        )

        psds[ch_name] = (
            freqs,
            psd
        )

        # ----------------------------------------------------
        # Absolute + log bandpower
        # ----------------------------------------------------

        bandpowers = {}

        for band_name, (
            low,
            high
        ) in BANDS.items():

            power = bandpower(
                freqs,
                psd,
                low,
                high
            )

            bandpowers[band_name] = power

            features[
                f"{ch_name}_{band_name}_power"
            ] = power

            features[
                f"{ch_name}_{band_name}_logpower"
            ] = safe_log(power)

        # ----------------------------------------------------
        # Total power
        # ----------------------------------------------------

        total_power = bandpower(
            freqs,
            psd,
            TOTAL_BAND[0],
            TOTAL_BAND[1]
        )

        features[
            f"{ch_name}_total_logpower"
        ] = safe_log(total_power)

        # ----------------------------------------------------
        # Relative bandpower
        # ----------------------------------------------------

        for band_name, power in bandpowers.items():

            relative_power = (
                power /
                max(total_power, 1e-12)
            )

            features[
                f"{ch_name}_{band_name}_relative"
            ] = relative_power

            # log relative bandpower
            features[
                f"{ch_name}_{band_name}_logrelative"
            ] = safe_log(
                relative_power
            )

        # ====================================================
        # HJORTH
        # ====================================================

        mobility, complexity = (
            hjorth_parameters(signal)
        )

        features[
            f"{ch_name}_hjorth_mobility"
        ] = mobility

        features[
            f"{ch_name}_hjorth_complexity"
        ] = complexity

        # ====================================================
        # APERIODIC
        # ====================================================

        slope, offset = (
            aperiodic_features(
                freqs,
                psd
            )
        )

        features[
            f"{ch_name}_aperiodic_slope"
        ] = slope

        features[
            f"{ch_name}_aperiodic_offset"
        ] = offset

        # ====================================================
        # SAMPLE ENTROPY
        # ====================================================

        # Convert to microvolts.
        # This makes r_ratio operate on a useful amplitude scale.
        signal_uv = signal * 1e6

        entropy = sample_entropy(
            signal_uv
        )

        features[
            f"{ch_name}_sample_entropy"
        ] = entropy

    # ========================================================
    # CHANNEL RATIOS / LOG DIFFERENCES
    # ========================================================

    # Log-power difference is equivalent to:
    #
    # log(P1) - log(P2)
    #
    # which is much more stable than raw ratios.

    for band_name in BANDS:

        fp1 = features[
            f"Fp1_{band_name}_logpower"
        ]

        fp2 = features[
            f"Fp2_{band_name}_logpower"
        ]

        fz = features[
            f"Fz_{band_name}_logpower"
        ]

        # Frontal asymmetry
        features[
            f"asymmetry_Fp2_minus_Fp1_{band_name}"
        ] = fp2 - fp1

        # Channel log ratios
        features[
            f"logratio_Fp1_Fz_{band_name}"
        ] = fp1 - fz

        features[
            f"logratio_Fp2_Fz_{band_name}"
        ] = fp2 - fz

    # ========================================================
    # PLV
    # ========================================================

    channel_pairs = [
        ("Fp1", "Fp2", 0, 1),
        ("Fp1", "Fz",  0, 2),
        ("Fp2", "Fz",  1, 2)
    ]

    for (
        ch1,
        ch2,
        idx1,
        idx2
    ) in channel_pairs:

        for band_name, (
            low,
            high
        ) in BANDS.items():

            plv = compute_plv(
                window[idx1],
                window[idx2],
                sfreq,
                low,
                high
            )

            features[
                f"PLV_{ch1}_{ch2}_{band_name}"
            ] = plv

    return features

In [28]:
all_rows = []

npz_files = sorted(
    PROCESSED_ROOT.glob(
        "sub-*_ses-session*.npz"
    )
)

print(
    "NPZ files found:",
    len(npz_files)
)

NPZ files found: 180


In [29]:
for file_idx, npz_file in enumerate(
    npz_files,
    start=1
):

    print(
        f"[{file_idx}/{len(npz_files)}] "
        f"{npz_file.name}"
    )

    # --------------------------------------------------------
    # Parse metadata
    # --------------------------------------------------------

    filename = npz_file.stem

    # Example:
    # sub-01_ses-session1

    parts = filename.split("_")

    subject = parts[0]
    session = parts[1]

    # --------------------------------------------------------
    # Load windows
    # --------------------------------------------------------

    data = np.load(
        npz_file
    )

    windows = data["windows"]

    sfreq = float(
        data["sfreq"]
    )

    print(
        "  windows:",
        windows.shape,
        "sfreq:",
        sfreq
    )

    # --------------------------------------------------------
    # Extract features
    # --------------------------------------------------------

    for window_idx, window in enumerate(
        windows
    ):

        features = extract_features(
            window,
            sfreq
        )

        features["subject"] = subject
        features["session"] = session
        features["window"] = window_idx

        all_rows.append(
            features
        )


features_df = pd.DataFrame(
    all_rows
)

print(
    "Final feature matrix:",
    features_df.shape
)

[1/180] sub-01_ses-session1.npz
  windows: (69, 3, 2000) sfreq: 500.0


[2/180] sub-01_ses-session2.npz
  windows: (73, 3, 2000) sfreq: 500.0
[3/180] sub-01_ses-session3.npz
  windows: (56, 3, 2000) sfreq: 500.0
[4/180] sub-02_ses-session1.npz
  windows: (33, 3, 2000) sfreq: 500.0
[5/180] sub-02_ses-session2.npz
  windows: (41, 3, 2000) sfreq: 500.0
[6/180] sub-02_ses-session3.npz
  windows: (1, 3, 2000) sfreq: 500.0
[7/180] sub-03_ses-session1.npz
  windows: (15, 3, 2000) sfreq: 500.0
[8/180] sub-03_ses-session2.npz
  windows: (11, 3, 2000) sfreq: 500.0
[9/180] sub-03_ses-session3.npz
  windows: (44, 3, 2000) sfreq: 500.0
[10/180] sub-04_ses-session1.npz
  windows: (53, 3, 2000) sfreq: 500.0
[11/180] sub-04_ses-session2.npz
  windows: (70, 3, 2000) sfreq: 500.0
[12/180] sub-04_ses-session3.npz
  windows: (54, 3, 2000) sfreq: 500.0
[13/180] sub-05_ses-session1.npz
  windows: (44, 3, 2000) sfreq: 500.0
[14/180] sub-05_ses-session2.npz
  windows: (12, 3, 2000) sfreq: 500.0
[15/180] sub-05_ses-session3.npz
  windows: (47, 3, 2000) sfreq: 500.0
[16/180] sub-06

In [30]:
# Put metadata first

metadata_columns = [
    "subject",
    "session",
    "window"
]

feature_columns = [
    c for c in features_df.columns
    if c not in metadata_columns
]

features_df = features_df[
    metadata_columns +
    feature_columns
]

features_df.to_csv(
    OUTPUT_FILE,
    index=False
)

print(
    "Saved:",
    OUTPUT_FILE
)

print(
    "Number of features:",
    len(feature_columns)
)

Saved: C:\Users\Sanjeev\OneDrive\Desktop\Brain ID\EEG\ds004148\processed_windows\combined_eeg_features.csv
Number of features: 108
